# Reporting and Visualisation - A&E Department 

The notebook has data reporting and visualisation, for a synthetic A&E Department

It covers different questions that are useful to understanding KPIs within an A&E Department:

1) Patient Flow -> What are the average minutes across a patient's lifecycle, stats are inspired by NHS statistics 
2) Left before being seeing -> The proportion of patients who leave before being seen, with their triage category.
3) Triage demand -> What occupies the most time in the department?

View Interactive Version via NBViewer: 

## Setup

In [1]:
from sqlalchemy import create_engine
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.io as pio
import os
from dotenv import load_dotenv


pio.renderers.default = "notebook_connected"

In [2]:
# Database Connections
load_dotenv()

conn_string = os.getenv('NEON_URL')
engine = create_engine(conn_string)


## Analysis 

### Patient Flow

**Objective:**

 Calculate the mean duration of three distinct phases during a patient journey, grouped by triage category. 

**Observations:**

- Clear inverse relationship between triage ccategory and time to first assessment. Life threatening conditions are assessed within 2.36 minutes, non urgent cases wait an average of 19 minutes

- Triage system is sucessfully prioritising high-risk patients, but there's a bottlneck between assessment and treatment start between all categories, showing 60-85 minutes wait

In [3]:
# Picks the triage name, calculates the average number of minutes from x timestamp to y timesamp

query_1 = '''
SELECT
    t.triage_name,
    ROUND(AVG(EXTRACT(EPOCH FROM (v.first_assessment_time - v.arrival_time)) / 60), 2) AS avg_mins_to_first_assessment,
    ROUND(AVG(EXTRACT(EPOCH FROM (v.treatment_start - v.arrival_time)) / 60), 2) AS avg_mins_to_treatment_start,
    ROUND(AVG(EXTRACT(EPOCH FROM (v.discharge_time - v.arrival_time)) / 60), 2) AS avg_total_visit_mins,
    COUNT(*) AS visit_count
FROM visit v
JOIN triage_category t
    ON v.triage_id = t.triage_id
WHERE v.first_assessment_time IS NOT NULL
GROUP BY t.triage_name
ORDER BY avg_mins_to_first_assessment DESC;
'''

query_1_df = pd.read_sql(query_1, engine)
query_1_df

,triage_name,avg_mins_to_first_assessment,avg_mins_to_treatment_start,avg_total_visit_mins,visit_count
0,Not Urgent,19.03,85.25,239.33,28737
1,Not Threatening to Life & Limb,13.23,81.39,239.81,78597
2,Urgent,9.27,78.25,239.04,35995
3,Very Urgent,4.32,77.20,245.21,721
4,Life Threatening Conditions,2.36,66.77,217.68,22


In [4]:
fig = go.Figure()

# Stages of a patient flow 


fig.add_trace(go.Bar(
    x=query_1_df['triage_name'],
    y=query_1_df['avg_mins_to_first_assessment'],
    name='First Assessment',
    marker_color='lightslategray', 
    opacity=0.4
))

fig.add_trace(go.Bar(
    x=query_1_df['triage_name'],
    y=query_1_df['avg_mins_to_treatment_start'] - query_1_df['avg_mins_to_first_assessment'],
    name='Assessment to Treatment',
    marker_color='slategrey', 
    opacity=0.7
))

fig.add_trace(go.Bar(
    x=query_1_df['triage_name'],
    y=query_1_df['avg_total_visit_mins'] - query_1_df['avg_mins_to_treatment_start'],
    name='Treatment to Discharge',
    marker_color='darkslategrey',
    opacity=0.9
))

fig.update_layout(
    title='<b>Average Patient Journey Duration</b>',
    xaxis_title='Triage Category',
    yaxis_title='Minutes from Arrival',
    barmode='stack',
    template='plotly_white',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=dict(t=100)
)

fig.show()

### Left before being seen

**Objective:**

Determine the volume and percentage of patients who left early

**Observations:**

- The rate is the highest in the Not Threatening to Life and Limb and Urgent categories. The percentages across the board are low, but the volume is high. 

- High rates in the Urgent (2.3%) catgory is a red flag as it suggests they are getting frustrated with the waiting times potentially, whilst also needing attention due to their triage category, which could lead to negative effects

In [5]:
query_2 = '''
SELECT 
    t.triage_name,
    COUNT(*) AS total_patients,
    COUNT(CASE WHEN v.treatment_start IS NULL THEN 1 END) AS left_before_treatment,
    ROUND(100.0 * COUNT(CASE WHEN v.treatment_start IS NULL THEN 1 END) / COUNT(*), 1) AS pct_left_early
FROM visit v
JOIN triage_category t ON v.triage_id = t.triage_id
GROUP BY t.triage_name
ORDER BY pct_left_early DESC;
'''

query_2_df = pd.read_sql(query_2, engine)
query_2_df

,triage_name,total_patients,left_before_treatment,pct_left_early
0,Urgent,35995,832,2.3
1,Not Threatening to Life & Limb,78597,1719,2.2
2,Very Urgent,721,15,2.1
3,Not Urgent,28737,614,2.1
4,Life Threatening Conditions,22,0,0.0


In [6]:
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Total patient volume
fig.add_trace(
    go.Bar(
        x=query_2_df['triage_name'],
        y=query_2_df['total_patients'],
        name="Total Patients",
        marker_color='lightslategray',
        opacity=0.6,
        hovertemplate="Category: %{x}<br>Total Patients: %{y}<extra></extra>"
    ),
    secondary_y=False,
)

# 2.  percentage who left early
fig.add_trace(
    go.Scatter(
        x=query_2_df['triage_name'],
        y=query_2_df['pct_left_early'],
        name="% Left Before Treatment",
        mode='lines+markers',
        marker=dict(size=10, color='crimson'),
        line=dict(width=3),
        hovertemplate="Category: %{x}<br>Left Early: %{y}%<extra></extra>"
    ),
    secondary_y=True,
)

fig.update_layout(
    title='<b>Patients -  Left Before Treatment by Triage Category</b>',
    xaxis_title='Triage Category',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    template='plotly_white',
    hovermode="x unified"
)

# Set y-axes titles
fig.update_yaxes(title_text="Number of Patients", secondary_y=False)
fig.update_yaxes(title_text="Percentage (%) Left Early", secondary_y=True, range=[0, query_2_df['pct_left_early'].max() + 5])

fig.show()

### Triage Demand

**Objective:**

Calculate total department hours by multiplying visit counts by summing the difference in discharge and arrival

**Observations:**

- Despite life threatening cases being the most intense, they represent the smallest portion of hours. The bulk is driven by not threatening to life and limb. 
- This could call for offloading the not threatening category to a separate stream, so the main department can more resources on the higher priority cases

In [7]:
query_3 = '''
SELECT
    t.triage_name,
    COUNT(*) AS visit_count,
    ROUND(AVG(EXTRACT(EPOCH FROM (v.discharge_time - v.arrival_time)) / 60), 1) AS avg_total_visit_mins,
    ROUND(SUM(EXTRACT(EPOCH FROM (v.discharge_time - v.arrival_time)) / 3600), 1) AS total_department_hours
FROM visit v
JOIN triage_category t
    ON v.triage_id = t.triage_id
WHERE v.discharge_time IS NOT NULL
GROUP BY t.triage_name
ORDER BY total_department_hours DESC;
'''

query_3_df = pd.read_sql(query_3, engine)
query_3_df

,triage_name,visit_count,avg_total_visit_mins,total_department_hours
0,Not Threatening to Life & Limb,78597,239.8,314139.9
1,Urgent,35995,239.0,143404.9
2,Not Urgent,28737,239.3,114625.8
3,Very Urgent,721,245.2,2946.6
4,Life Threatening Conditions,22,217.7,79.8


In [8]:
fig = px.treemap(
    query_3_df, 
    path=[px.Constant('Total A&E Workload'), 'triage_name'], 
    values='total_department_hours',
    color='total_department_hours',
    color_continuous_scale='Greys', 
    title='<b>Workload Distribution (Hours)</b>',
    template='plotly_white',
)

fig.update_traces(
    textinfo='label+value+percent parent',
    texttemplate='<b>%{label}</b><br>%{value:.1f} Hours<br>%{percentParent:.1%}',
    marker=dict(line=dict(width=1, color='white')) 
)

fig.update_layout(
    coloraxis_showscale=True,
    margin=dict(t=50, l=10, r=10, b=10)
)

fig.show()